# 第1章 5分でLLMと話す

コードの仕組みを分解しながら、Google Colabで本物のLLMを0円で動かす

## このノートブックの使い方・学習目標

- **教科書のWebページ**：https://sakunokyumei.github.io/llm-research-textbook/01-first-llm.html
- **エラーが出たとき**：[エラー辞典（付録E）](https://sakunokyumei.github.io/llm-research-textbook/e-errors.html)
- **復習・理解度管理**：[学習状況・間隔復習ダッシュボード](https://sakunokyumei.github.io/llm-research-textbook/review.html)
- **この章でつくるもの**：ブラウザだけで動く自分専用のチャットLLM（Qwen2.5-0.5B-Instruct）と、入れ子データ・パラメータ数を扱う基本関数
- **身につく数学・理論**：LLMは「次の言葉の確率」を予測し続ける機械 ／ チャットの正体は「台本の続き」を書かせること ／ パラメータ（調整できる数）と「大規模」・メモリ見積もりの意味
- **新しく使うPython・PyTorch**：セルの実行・変数・文字列 ／ モジュールと `import` / `from ... import` ／ クラス・オブジェクト・メソッド・属性 ／ キーワード引数と条件式 `A if 条件 else B` ／ リスト `[ ]` と辞書 `{ }` の入れ子 ／ 関数 `def` と初期値つき引数 ／ `for` ループと `sum(... for ...)`

## 実行環境・リソースの目安

| 項目 | 目安 |
|---|---|
| 推奨ランタイム | Google Colab（GPU：T4） |
| 所要時間目安 | 60〜90分 |
| 難易度 | ★☆☆☆☆ |

> **⚠️ GPUランタイム設定**：メニューの「ランタイム」→「ランタイムのタイプを変更」→「**T4 GPU**」を選んでから上から順に実行してください。
> 無料枠の混雑・上限でGPUに接続できない場合は、時間をおくか [Kaggle Notebooks](https://sakunokyumei.github.io/llm-research-textbook/a-setup.html) を使うか、各章のCPU短縮手順で進められます。接続切断に備え、学習済みモデルはこまめにGoogleドライブへ保存しましょう。

> **🎯 この章でつくるもの**
>
> この章が終わるころ、あなたは次の3つができるようになります。
>
> 1. 無料のGoogle Colab上で**本物のLLM（Qwen2.5-0.5B-Instruct）**を動かし、キャラクター設定を変えて会話できる
> 2. LLM読み込みコードや返り値 `result[0]["generated_text"][-1]["content"]` の**入力・中間値・出力・データ型**を自分の言葉で説明できる
> 3. LLMがなぜ「次の言葉の予測機」なのか、なぜ掛け算を間違えやすいのかを理由つきで説明できる

**ミナト**：Pythonの基本（変数や `if` や `for`、関数）は少し書いたことがあるんですけど、AIのライブラリが出てくると急に暗号みたいに見えてしまって……。

**ボンゴ先生**：その感覚、とてもまっとうだよ。普通のPythonが書けても、`transformers` や `torch` のコードが難しく見えるのは当然なんだ。理由はだいたい3つ──**「ライブラリとオブジェクトの書き方」「リストと辞書の深い入れ子」「1行にまとめた短縮記法」**がいっぺんに出てくるからだ。<br>だからこの章では、「呪文だと思って貼って終わり」にはしない。本物のLLMコードの前に、**理解のための小さなPythonコード**で部品を1つずつ確かめてから進もう！

## Google Colabを開く

LLMを動かすにはたくさんの計算ができるコンピュータが必要ですが、Googleが無料で貸してくれる **Google Colaboratory**（略して **Colab**、コラボ）を使います。あなたのパソコンにPythonを入れなくても、ブラウザからGoogleのコンピュータ上でPythonを実行できます。

1. ブラウザで <https://colab.research.google.com/> を開きます
2. Googleアカウントでログインします（右上の「ログイン」）
3. **「ノートブックを新規作成」**（または左上メニュー「ファイル」→「ノートブックを新規作成」）を押します

表示された灰色の入力欄を **セル**（cell）と呼びます。ここにPythonコードを書いて実行します。

> **✨ コツ：ノートブックに名前をつけよう**
>
> 左上の「Untitled0.ipynb」をクリックして `01-first-llm.ipynb` のように名前を変えておくと、あとで探しやすくなります。ノートブックはGoogleドライブの「Colab Notebooks」フォルダに自動保存されます。

## 最初の1行を実行する

セルに次のコードを入力し、左の **▶（再生ボタン）** または <kbd>Shift</kbd> + <kbd>Enter</kbd> で実行します（最初の1回だけ接続に数秒〜十数秒かかります）。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
print("こんにちは、LLM！")

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
こんにちは、LLM！
```

</details>

新しいセルは **「+ コード」** で追加できます。変数と文字列の動きも1つ確かめましょう。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
greeting = "ボン"
print(greeting * 3 + "ゴ！")

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
ボンボンボンゴ！
```

</details>

<details><summary><b>基礎の補足：変数・文字列・セルの実行順（基礎確認）</b>（クリックで開く）</summary>

- `greeting = "ボン"` の `=` は数学の「等しい」ではなく、**「右の値を、左の名前の変数に入れる（代入）」**という意味です（等しいか比べるのは `==`）。
- 入力：`"ボン"`（データ型 `str`＝文字列）。`greeting * 3` の中間値は `"ボンボンボン"`（`str`）、それに `"ゴ！"` を `+` でつなげた出力が `"ボンボンボンゴ！"` です。
- **セルの実行順**：変数は「実行したセル」で初めて作られます。下のセルだけ実行して `NameError: name 'greeting' is not defined` と出たら、上のセルを先に実行してください。メニューの「ランタイム」→「すべてのセルを実行」で上から全部やり直せます。

</details>

#### 📝 確認問題（`ch01-q01` · Lv.1 · 変数・文字列演算とセル実行順）

次のPythonコードを実行すると、画面に何が表示されますか？

```python
word = "AI"
msg = word * 2 + "!"
print(msg)
```

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：Pythonで文字列に対して `*`（掛け算）と `+`（足し算）を使うと、それぞれどんな操作になるか思い出しましょう。
- **ヒント 2**：`word * 2` で `"AI"` を2回くり返した文字列を作り、その末尾に `"!"` をつなげます。
- **期待される出力**：`AIAI!`

**考え方**：Pythonでは文字列に整数を掛けると反復され、`+` で別の文字列と連結されます。

1. `word` に文字列 `"AI"` を代入する。
1. `word * 2` は `"AIAI"` になる。
1. `"AIAI" + "!"` は `"AIAI!"` になり、`print(msg)` でそのまま出力される。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/01-first-llm.html#ex-ch01-q01

</details>

## 無料のGPUを借りる

LLMの計算を速くするため、**GPU**（ジーピーユー）という部品を無料で借ります。GPUはもともとゲーム画面を描くための部品で、「画面の何百万個もの点の色を一斉に計算する」のが得意です。LLMも「何億個もの数の掛け算・足し算を一斉に計算する」処理が中心なので、GPUを使うと劇的に速くなります。

1. メニューの **「ランタイム」→「ランタイムのタイプを変更」** を開く
2. 「ハードウェア アクセラレータ」で **「T4 GPU」** を選ぶ
3. **「保存」** を押す

> **⚠️ つまずき注意：「GPUに接続できません」と言われたら**
>
> 無料枠の混雑や上限で借りられないときは、**CPUのままで大丈夫**です。この章のモデル（0.5B）は小さいのでCPUでも動きます。GPUがどうしても欲しい日は[付録A](https://sakunokyumei.github.io/llm-research-textbook/a-setup.html)の「Kaggle Notebooks」も無料で使えます。

借りられたか確認します。行頭の `!` は「PythonではなくLinuxコマンドとして実行して」という合図です。

🔎 **実行前に確認**：コードの各行が何を受け取り、何を作るか確認してから ▶ を押してください。

In [ ]:
!nvidia-smi

<details><summary><b>📊 出力の例（数値や文章は実行ごとに変わってOK）</b>（実行後に自分の出力と見比べよう）</summary>

```text
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
+-----------------------------------------+------------------------+----------------------+
```

</details>

`Tesla T4` と `15360MiB`（約15GB）が見えれば成功です。この15GBはGPU専用メモリ（**VRAM**）で、モデルを載せる机の広さにあたります。CPUのままの場合は `command not found` になりますが気にせず進みましょう。

#### 📝 確認問題（`ch01-q02` · Lv.2 · GPUを使う理由とVRAM）

LLMを動かすときに、CPUではなくGPUを使うと速くなる一番の理由はどれでしょう？

- **a.** GPUがない環境ではPythonの `print` や `def` といった基本文法がそもそも実行できないから
- **b.** GPUを使うとモデルのダウンロードファイル容量が自動で10分の1ほどに圧縮されるから
- **c.** GPUは大量の単純な数値計算を「一斉に（並列に）」実行する多数のコアを備えているから
- **d.** GPUの内部には日本語の辞書やトークナイザーが最初からハードウェアとして内蔵されているから

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：GPUがもともと「画面の何百万個もの点の色」を計算するために作られたことを思い出しましょう。
- **ヒント 2**：少数の高性能コアで順番にこなすCPUに対し、多数の小さなコアで同時並行に計算するGPUの強みを選びます。
- **正解**：`c`

**考え方**：LLMの推論や学習では、何億個ものパラメータに対する掛け算・足し算を同時に行います。GPUは何千もの小さな計算コアでそれを一斉に処理できます。

1. CPUは複雑な処理を手順どおりにこなすのが得意（少数の高性能な頭脳）。
1. GPUは単純な計算を大量に同時実行するのが得意（多数の計算係）。
1. LLMは大量の数値計算の塊なので、GPUと相性が非常に良い。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/01-first-llm.html#ex-ch01-q02

</details>

## LLMを読み込む前のPython基礎

次の節で本物のLLMを読み込むコードを書きますが、その前に**「なぜそういう書き方をするのか」**を最小限のPythonコードで分解しておきましょう。ここを押さえておくと、PyTorchや `transformers`（Hugging FaceのLLM用ライブラリ）のコードが一気に読めるようになります。

<details><summary><b>基礎の補足：理解のための最小コード：モジュール・クラス（設計図）・オブジェクト・メソッド・属性</b>（クリックで開く）</summary>

AIのライブラリでは、**「データ」と「そのデータに対する操作」を1つの箱にまとめたもの（オブジェクト）**をたくさん使います。小さな例で動きを見てみましょう。

```python no-run
# 1. モジュール／ライブラリを借りる
import math                   # math という道具箱まるごとを借りる → math.sqrt(9) のように使う
from math import sqrt         # 道具箱から sqrt だけ取り出す → sqrt(9) と短く書ける

# 2. クラス（設計図）とオブジェクト（実物）
class MiniBot:
    def __init__(self, name):
        self.name = name        # 属性（オブジェクトが持つデータ）
    def greet(self, user):      # メソッド（オブジェクトが持つ動作）
        return f"{self.name}：{user}さん、こんにちは"
    def __call__(self, text):   # オブジェクトを関数のように bot("...") と呼べる仕組み
        return f"{self.name}の返事：{text}だね！"

bot = MiniBot("ボンゴ")          # 設計図から実物（オブジェクト）を1つ作る
print(type(bot))                # <class '__main__.MiniBot'>
print(bot.name)                 # 属性（データなので () なし）→ ボンゴ
print(bot.greet("ミナト"))      # メソッド（動作なので () あり）→ ボンゴ：ミナトさん、こんにちは
print(bot("いい天気"))          # __call__ により関数のように呼べる → ボンゴの返事：いい天気だね！
```

| 用語 | 見分け方 | 意味 | LLMコードでの登場例 |
|---|---|---|---|
| **モジュール／ライブラリ** | `import torch` | 便利な道具をまとめた箱 | `torch`、`transformers` |
| **オブジェクト** | 変数に入っている実物 | データと機能を持つ部品 | `chat`、`chat.model`、`chat.tokenizer` |
| **属性（attribute）** | `モノ.名前`（`()` なし） | オブジェクトの中のデータ | `chat.model`、`chat.tokenizer`、`model.device` |
| **メソッド（method）** | `モノ.名前(...)`（`()` あり） | オブジェクトにさせる処理 | `torch.cuda.is_available()`、`p.numel()` |
| **キーワード引数** | `名前=値` | 引数の意味を明示して渡す | `model="..."`、`max_new_tokens=100` |
| **条件式（一行if）** | `A if 条件 else B` | 条件が真なら `A`、偽なら `B` | `torch.float16 if gpu else torch.float32` |

`chat = pipeline(...)` で作られる `chat` は、上の `bot` と同じように**「中に部品（`chat.model` や `chat.tokenizer`）を持ちつつ、自分自身を `chat(messages)` と関数のように呼べるオブジェクト」**です。

</details>

確認問題で、今の文法を自分の言葉で確かめましょう。

#### 📝 確認問題（`ch01-q03` · Lv.1 · モジュール・オブジェクト・メソッド・属性の区別）

次のコードで、`bot.name` と `bot.greet("ミナト")` の違いとして正しいものはどれですか？

```python
class MiniBot:
    def __init__(self, name):
        self.name = name
    def greet(self, user):
        return f"{self.name}：{user}さん、こんにちは"

bot = MiniBot("ボンゴ")
```

- **a.** `bot.name` も他のメソッドと同じように、必ず後ろに `()` を付けて `bot.name()` と呼び出す必要がある
- **b.** `bot.name` はオブジェクトが持つデータ（属性）、`bot.greet("ミナト")` はオブジェクトが持つ動き（メソッド）の呼び出し
- **c.** `bot.name` が処理を実行するメソッドであり、`bot.greet("ミナト")` がデータを格納した属性である
- **d.** `bot.name` も `bot.greet(...)` も、事前に `import` しないと使えない外部ライブラリの関数である

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：後ろに丸かっこ `(...)` が付いているかどうかで、データと動作を区別できます。
- **ヒント 2**：「データ（名札）」は `()` なしの属性、「動作（命令）」は `()` ありのメソッドです。
- **正解**：`b`

**考え方**：PyTorchやTransformersのコードでは、`chat.model`（属性＝中身の部品）や `chat.model.parameters()`（メソッド＝部品からデータを取り出す処理）のように `.` の後ろが頻繁に登場します。

1. `bot` は `MiniBot` 設計図（クラス）から作られた実物（オブジェクト）。
1. `bot.name` は中に入っているデータ（属性）なので `()` なしで `"ボンゴ"` が取り出せる。
1. `bot.greet("ミナト")` は実物に紐づく関数（メソッド）なので `()` で引数を渡して実行する。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/01-first-llm.html#ex-ch01-q03

</details>

#### 📝 確認問題（`ch01-q04` · Lv.2 · 条件式（一行if文）の読み方）

LLM読み込みコードに出てきた `A if 条件 else B` の練習です。次のコードの出力を予測してください。

```python
gpu = False
dtype_name = "float16" if gpu else "float32"
print(dtype_name)
```

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：`A if 条件 else B` は条件が真なら左側、偽なら右側を返す1行の条件式です。
- **ヒント 2**：ここでは `gpu` が `False` なので、`else` の右側にある文字列が `dtype_name` に入ります。
- **期待される出力**：`float32`

**考え方**：`A if 条件 else B` は条件が真（`True`）なら `A`、偽（`False`）なら `B` を返す式です。

1. `gpu` は `False` である。
1. `"float16" if False else "float32"` を評価すると `else` 側が選ばれる。
1. `dtype_name` は `"float32"` となり、出力は `float32`。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/01-first-llm.html#ex-ch01-q04

</details>

## LLMを呼び出す

それでは、本物のLLMを読み込む**実用コード**を実行します。各行の入力・出力・目的をすぐ下の表と照らし合わせながら確認してください。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
import torch
from transformers import pipeline

gpu = torch.cuda.is_available()   # GPUが使えるなら True、使えないなら False

chat = pipeline(
    "text-generation",                         # やりたいこと：文章を生成する
    model="Qwen/Qwen2.5-0.5B-Instruct",        # 使うモデルの名前
    dtype=torch.float16 if gpu else torch.float32,  # 数の細かさ（GPUなら軽い形式）
    device_map="auto",                         # GPUがあればGPUに載せる
)
print("GPUを使う？", gpu)

<details><summary><b>📊 出力の例（数値や文章は実行ごとに変わってOK）</b>（実行後に自分の出力と見比べよう）</summary>

```text
config.json: 100%|██████████| 659/659 [00:00<00:00, 4.21MB/s]
model.safetensors: 100%|██████████| 988M/988M [00:08<00:00, 118MB/s]
...
Device set to use cuda:0
GPUを使う？ True
```

</details>

このコードで何が起きているかを整理します。

| コード | 入力／値と型 | 出力／結果 | なぜ必要か |
|---|---|---|---|
| `torch.cuda.is_available()` | なし | `True` または `False`（`bool`） | GPUがあるか判定し、数値の形式を切り替えるため |
| `"text-generation"` | `str` | タスク指定 | `pipeline` に「文章の続きを作る仕事」と伝えるため |
| `model="Qwen/Qwen2.5-0.5B-Instruct"` | `str` | Hugging Faceから約1GB取得 | 公開モデルを指定してダウンロード・読み込みするため |
| `dtype=torch.float16 if gpu else torch.float32` | `torch.dtype` | 1個あたり2Bまたは4B | GPUでは半分のメモリで速く動く `float16`（16ビット小数、第13章で詳しく）を使い、CPUでは安定な `float32` にするため |
| `device_map="auto"` | `str` | GPU（`cuda:0`）かCPUへ配置 | 手動で置き場所を書かなくても自動で最適な機器に載せるため |
| `chat = pipeline(...)` | 上の引数一式 | `TextGenerationPipeline` オブジェクト | トークナイザー・モデル本体・生成ループを1つにまとめて扱いやすくするため |

モデル名の `Qwen/Qwen2.5-0.5B-Instruct` は、Alibaba社のチームが公開したモデルで、`Qwen`（チェン）がシリーズ名、`2.5` が世代、`0.5B` がパラメータ数（約5億個）、`Instruct` は「指示や会話に従う追加学習済み」の意味です（商用も可能なApache 2.0ライセンス）。

> **✨ コツ：環境・リソースとライブラリ版の目安**
>
> - **実行環境**：Colab無料 T4 GPU を推奨（約10〜30秒で読み込み、応答は数秒）。CPUでも実行可能（読み込み約1分、応答は十数秒〜数十秒）。
> - **ダウンロードとメモリ**：初回のみモデル重み約 1 GB をダウンロード。VRAM（またはRAM）は `float16` で約 1〜2 GB、`float32`（CPU）で約 2〜3 GB 使用します。
> - **Colab切断**：放置で切断されてもコードは残るため、再接続して上から再実行すれば復旧できます。
> - **ライブラリ版**：上の `dtype=` 引数は `transformers >= 4.56`（5.x含む、Colab標準）で対応しています。手元PCなどで古い版を使い `TypeError: unexpected keyword argument 'dtype'` が出た場合は、`!pip install -U transformers` で更新するか `torch_dtype=` に読み替えてください（動作確認済みの版一覧はリポジトリの `requirements-book.txt` と[付録A](https://sakunokyumei.github.io/llm-research-textbook/a-setup.html)を参照）。

> **⚠️ つまずき注意：赤や黄色の文字が出ても、まず落ち着く**
>
> `The secret HF_TOKEN does not exist in your Colab secrets.` という黄色い警告が出ることがありますが、公開モデルなのでログインなしで問題ありません。最後の行に `GPUを使う？ True`（または `False`）が出ていれば成功です。

#### 📝 確認問題（`ch01-q05` · Lv.2 · pipeline()の引数と役割）

`pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct", dtype=torch.float16 if gpu else torch.float32, device_map="auto")` の各引数の役割として、**誤っている**ものはどれですか？

- **a.** `"text-generation"` は「入力の続きを生成する文章生成タスクを行う」という指定である
- **b.** `dtype=...` はモデルのパラメータ数値を16ビット小数か32ビット小数かで持つ指定である
- **c.** `device_map="auto"` はGPUが使えればGPUに、なければCPUに自動で配置する指定である
- **d.** `model="Qwen/Qwen2.5-0.5B-Instruct"` はAIに最初の一言として投げかける質問文である

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：`pipeline(...)` はモデルを「準備する段階」で、まだ会話の質問は渡していません。
- **ヒント 2**：`model=...` に書かれている `"Qwen/Qwen2.5-0.5B-Instruct"` が何の名前かを考えましょう。
- **正解**：`d`

**考え方**：`model=...` はHugging Face上で公開されている学習済みモデルを特定するための識別名（リポジトリID）です。

1. `pipeline(...)` の時点では「どのモデルをどう準備するか」だけを決めて `chat` オブジェクトを作る。
1. `model="Qwen/Qwen2.5-0.5B-Instruct"` は「作成者/モデル名」の形式。
1. 質問文（プロンプト）は準備後に `chat(messages)` で渡す。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/01-first-llm.html#ex-ch01-q05

</details>

## 話しかけてみる

いよいよ会話します。ただし、`chat(...)` に渡す `messages` と、返ってくる `result` は**リスト `[ ]` と辞書 `{ }` の入れ子**になっています。いきなり本物を見る前に、小さなダミーデータで「どの箱を順番に開けているか」を確かめましょう。

<details><summary><b>基礎の補足：理解のための最小コード：入れ子の箱を1枚ずつ開ける</b>（クリックで開く）</summary>

```python no-run
# chat(messages) が返す result の正体は、このような「リストと辞書の入れ子」です
dummy_result = [
    {
        "generated_text": [
            {"role": "user", "content": "こんにちは！"},
            {"role": "assistant", "content": "こんにちは！お手伝いします。"},
        ]
    }
]

step1 = dummy_result[0]                  # list の 0番目 → dict
step2 = step1["generated_text"]          # dict の値      → list（会話の履歴）
step3 = step2[-1]                        # list の 末尾   → dict（最新のAIの返事）
step4 = step3["content"]                 # dict の値      → str（文章の中身）

print(type(step1), type(step2), type(step3), type(step4))
# <class 'dict'> <class 'list'> <class 'dict'> <class 'str'>
print(step4)
# こんにちは！お手伝いします。
```

これら4段階を1行につなげたものが `result[0]["generated_text"][-1]["content"]` です。**なぜ `[-1]`（後ろから1番目）を使うのか？** それは、会話履歴の先頭には `system` や `user` の発話が入って長さが変わる一方、AIの最新の返事は常に**一番最後**に付け足されるからです。

</details>

仕組みが見えたところで、本物のLLMに話しかけてみましょう！

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
messages = [
    {"role": "user", "content": "こんにちは！あなたは誰ですか？"},
]
result = chat(messages, max_new_tokens=100)
print(result[0]["generated_text"][-1]["content"])

<details><summary><b>📊 出力の例（数値や文章は実行ごとに変わってOK）</b>（実行後に自分の出力と見比べよう）</summary>

```text
こんにちは！私はQwenです。Alibaba Cloudによって開発された大規模言語モデルです。質問に答えたり、文章を書いたり、いろいろなお手伝いができます。何かお手伝いできることはありますか？
```

</details>

**ボンゴ先生**：動いた！ 今、キミのブラウザの向こうで、約5億個の数がいっせいに計算されて、この返事が生まれたんだ。拍手！ 🥁ドコドコ

答えは実行するたびに少し変わります。LLMは毎回、確率に従ってサイコロを振って言葉を選んでいるからです（このサイコロの仕組みと数式は第3章でゼロから自作します）。

#### 📝 確認問題（`ch01-q06` · Lv.3 · リストと辞書の入れ子構造の読み解き）

`chat(messages)` の返り値と同じ入れ子構造です。次のコードの出力を予測してください。

```python
result = [
    {
        "generated_text": [
            {"role": "user", "content": "こんにちは"},
            {"role": "assistant", "content": "やあ！"}
        ]
    }
]
print(result[0]["generated_text"][-1]["content"])
```

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：入れ子になったリストと辞書を、左のインデックスから1段階ずつ外側→内側へたどっていきましょう。
- **ヒント 2**：`result[0]`（辞書）→ `["generated_text"]`（2件の会話リスト）→ `[-1]`（最後の要素＝assistantの辞書）→ `["content"]` の順に開きます。
- **期待される出力**：`やあ！`

**考え方**：`result` は「1件の生成結果（辞書）が入ったリスト」で、その中の `"generated_text"` に会話全体（辞書のリスト）が入っています。最新の返答は常に末尾 `[-1]` にあります。

1. `result[0]` は `{"generated_text": [...]}` という辞書。
1. `result[0]["generated_text"]` は長さ2のリスト（0番目がuser、1番目がassistant）。
1. `[-1]` で末尾（assistantの辞書）を取り出す。
1. `["content"]` で `"やあ！"` を取り出す。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/01-first-llm.html#ex-ch01-q06

</details>

#### 📝 確認問題（`ch01-q07` · Lv.2 · なぜ [-1] を使うのか）

会話履歴からAIの最新の返事を取り出すとき、`[1]` ではなく `[-1]` を使う方が安全な理由はどれでしょう？

- **a.** リストに対して固定番号の `[1]` と書くと、要素数にかかわらず必ず `IndexError` が起きるから
- **b.** Pythonではリストの先頭番号が `0` ではなく `-1` から始まる仕様になっているから
- **c.** `[-1]` を指定するとリスト全体が逆順に並べ替えられ、最新の要素が先頭へ移動するから
- **d.** リスト長が `system` 追加や会話往復で変わっても、最新の返事は常に末尾にあるから

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：会話リストに `system` メッセージが追加されたり、2往復目に入ったりすると要素数がどう変わるか考えましょう。
- **ヒント 2**：`[system, user, assistant]` の3件になったとき、`[1]` と `[-1]` はそれぞれ誰の発言を指すか数えてみてください。
- **正解**：`d`

**考え方**：固定の番号 `[1]` に頼ると、`system` プロンプトを追加したときや2往復目の会話で別のメッセージを取ってしまいます。`[-1]` なら常に最新（末尾）を取れます。

1. `[user, assistant]` のとき：`[1]` は assistant、`[-1]` も assistant。
1. `[system, user, assistant]` のとき：`[1]` は user（間違い！）、`[-1]` は assistant（正解！）。
1. よって長さが変わっても壊れない `[-1]` を使う。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/01-first-llm.html#ex-ch01-q07

</details>

## 毎回書くのは面倒なので、関数にする

質問のたびに4行書くのは手間なので、よく使う手順を **関数** `def` にまとめます。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
def ask(question, max_new_tokens=200):
    messages = [{"role": "user", "content": question}]
    result = chat(messages, max_new_tokens=max_new_tokens)
    return result[0]["generated_text"][-1]["content"]

print(ask("朝ごはんにおすすめの食べ物を3つ、理由つきで教えて"))

<details><summary><b>📊 出力の例（数値や文章は実行ごとに変わってOK）</b>（実行後に自分の出力と見比べよう）</summary>

```text
1. ご飯と味噌汁：エネルギーになる炭水化物と、体を温める味噌汁で一日を元気に始められます。
2. ヨーグルトとフルーツ：腸内環境を整え、ビタミンも取れます。
3. 卵料理：たんぱく質が豊富で、お腹が空きにくくなります。
```

</details>

- **入力**：`question`（`str`）、`max_new_tokens`（`int`、省略すると初期値 `200` が使われる）
- **出力**：AIの返事の文字列（`str`）
- **処理目的**：会話リストの組み立て → `chat(...)` 実行 → 最新返答の取り出し、という定型手順を1行で再利用できるようにする。

#### 🔬 やってみよう：いろいろ聞いてみよう（予想 → 実行 → 比較）

`ask("…")` の中身を変えて試してみましょう。実行する前に「得意そうか、苦手そうか」を予想してください。

- 得意なこと：「ITパスポートと基本情報技術者試験の違いを、高校生向けに説明して」
- 創作：「猫が主人公の、3行で終わる物語を書いて」
- 苦手かも：「123×456を計算して」

最後の掛け算、LLMの答えは合っていましたか？ Pythonの厳密な計算と比べましょう。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
print(123 * 456)

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
56088
```

</details>

#### 📝 確認問題（`ch01-q08` · Lv.2 · 関数の初期値つき引数）

次の関数 `make_limit` を2回呼び出します。表示される2行を予測してください。

```python
def make_limit(q, max_new_tokens=200):
    return f"{q}:{max_new_tokens}"

print(make_limit("天気"))
print(make_limit("天気", max_new_tokens=50))
```

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：関数定義の `引数=値`（デフォルト引数）は、呼び出し時に省略されたときだけ使われます。
- **ヒント 2**：1回目は第2引数を省略しているので初期値 `200`、2回目は `50` を明示しているので上書きされます。
- **期待される出力**：`天気:200
天気:50`

**考え方**：関数定義の `引数=値` はデフォルト引数（初期値）です。普段は省略して短く呼び、長さを変えたいときだけ指定できます。

1. 1回目：`make_limit("天気")` は `max_new_tokens` を省略したので初期値 `200` → `天気:200`。
1. 2回目：`max_new_tokens=50` を明示したので `50` が使われる → `天気:50`。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/01-first-llm.html#ex-ch01-q08

</details>

#### 📝 確認問題（`ch01-q09` · Lv.4 · 【自力実装】会話結果から最新返答を取り出す関数）

`pipeline` の返り値 `result` を受け取り、最新のAIの返答文字列（`content`）を返す関数 `extract_reply(result)` を自力で実装し、Colabで検算テストを通してください。

**ひな形コード：**
```python
def extract_reply(result):
    # ここに処理を書く
    # ヒント: result -> [0] -> ["generated_text"] -> [-1] -> ["content"]
    pass
```

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：`result` の入れ子構造を外側から1つずつ変数に取り出すと分かりやすくなります。
- **ヒント 2**：`result[0]["generated_text"]` で会話リストを取り出し、末尾 `[-1]` の辞書の `"content"` キーを返しましょう。

**解答例：**
```python
def extract_reply(result):
    conversation = result[0]["generated_text"]
    last_message = conversation[-1]
    return last_message["content"]
```

**検算テスト（Colabで実行して確認）：**
```python
sample1 = [{"generated_text": [{"role": "user", "content": "Hi"}, {"role": "assistant", "content": "Hello!"}]}]
sample2 = [{"generated_text": [{"role": "system", "content": "役"}, {"role": "user", "content": "Q"}, {"role": "assistant", "content": "A"}]}]
assert extract_reply(sample1) == "Hello!"
assert extract_reply(sample2) == "A"
print("ALL PASSED")
```

**考え方**：入れ子データを1段階ずつ名前付き変数に分解すると、型と意味を確認しながらバグなく実装できます。

1. `result[0]["generated_text"]` で会話リストを取り出す。
1. `conversation[-1]` で末尾のメッセージ辞書を取り出す。
1. `last_message["content"]` を `return` する。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/01-first-llm.html#ex-ch01-q09

</details>

## LLMの正体：次の言葉を当てるゲームの達人

なぜLLMは自然な日本語を書けるのに、`123×456` を間違えることがあるのでしょうか？ ここにLLMの一番大切な原理があります。

#### 💡 理論のポイント：LLMがやっていることは「次に来る言葉の確率予測」だけ

LLMが内部でやっているのは、**「ここまでの文章の続きとして、次に来そうな言葉ごとに確率をつけること」**だけです。

たとえば「吾輩は」という入力に対して、LLMは頭の中で候補ごとに確率を計算します（どの候補も0以上、全部足すとちょうど `1.0`＝100%）。

| 次の候補 | 確率（イメージ） |
|---|---|
| 猫 | `0.65`（65%） |
| 犬 | `0.15`（15%） |
| 人間 | `0.10`（10%） |
| その他の15万語 | 合計 `0.10`（10%） |
| **合計** | **`1.00`（100%）** |

1つ選んだら、その言葉を末尾に付け足して、また次を予測します。これを何百回も繰り返すと長い文章になります。

<div class="math-display">「吾輩は」→ 予測「猫」→「吾輩は猫」→ 予測「で」→「吾輩は猫で」→ 予測「ある」→ …</div>

つまり「123×456=」に対しても、LLMは筆算や電卓の計算をしているのではなく、**「次に来そうな数字の並び」を確率的に予測している**のです。だから小さなモデルほど、桁数や雰囲気だけはそれっぽいのに不正確な答えを出しやすいのです。

本当に「続きを書いているだけ」なのか確かめましょう。会話リスト `messages` ではなく、ただの文字列を渡してみます。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
out = chat("吾輩は猫である。名前は", max_new_tokens=30, do_sample=False)
print(out[0]["generated_text"])

<details><summary><b>📊 出力の例（数値や文章は実行ごとに変わってOK）</b>（実行後に自分の出力と見比べよう）</summary>

```text
吾輩は猫である。名前はまだ無い。どこで生れたかとんと見当がつかぬ。何でも薄暗いじめじめした所で
```

</details>

- **入力**：文字列 `"吾輩は猫である。名前は"`、`max_new_tokens=30`（最大30個のかけらを生成）、`do_sample=False`（サイコロを振らず、毎回いちばん確率が高い候補を選ぶ指定）
- **出力**：`out[0]["generated_text"]` はただの文字列（`str`）で、元の文＋予測された続きが返ります（文字列入力のときは会話リストではなく文字列がそのまま返る点にも注目！）。

#### 📝 確認問題（`ch01-q10` · Lv.1 · LLMの本質と確率の合計）

ある小さなLLMが「吾輩は」の次に来る候補を3つだけ考えており、確率が「猫：0.65」「犬：0.25」「机：？」だったとします。確率の合計は必ず `1.0` になるとき、「机」の確率はいくつですか？

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：確率分布では、すべての候補の確率を足し合わせるといくつになるか思い出しましょう。
- **ヒント 2**：全体の合計 `1.0` から、分かっている2つの確率 `0.65 + 0.25` を差し引いて求めます。
- **正解**：`0.1`

**考え方**：次に来るトークンの確率分布は、すべての候補の確率を合計すると必ず `1`（100%）になります。

1. 分かっている確率の合計：$0.65 + 0.25 = 0.90$
1. 全体は $1.00$ なので、残りは $1.00 - 0.90 = 0.10$

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/01-first-llm.html#ex-ch01-q10

</details>

#### 📝 確認問題（`ch01-q11` · Lv.2 · LLMが掛け算を間違えやすい根本理由）

小さなLLM（Qwen2.5-0.5Bなど）に「123×456を計算して」と聞くと、間違った数字を自信満々に答えることがあります。**なぜ**このようなことが起きるのか、LLMの仕組みから自分の言葉で説明してください。

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：LLMが「電卓のように厳密に計算しているのか」「別の原理で答えているのか」を対比させてみましょう。
- **ヒント 2**：「内部で筆算をしていない」「これまでの文章の続きとして確率的にそれっぽい次のトークンを1つずつ予測している」の2点を盛り込みます。
- **解答の目安**：LLMは内部で筆算や電卓のような厳密な四則演算をしているのではなく、「ここまでの文字の並びの続きとして、確率的にそれっぽい次のトークン」を1つずつ予測しているだけだからです。訓練データにそのまま載っていない掛け算では、桁数や雰囲気だけはそれっぽいけれど不正確な数字を予測してしまいます。

**考え方**：LLMの根本動作は「次のトークンの確率予測の繰り返し」であり、記号的な厳密計算エンジンではありません。

1. LLMは入力文の続きを確率が高い順に1トークンずつ生成する。
1. よく登場する計算（例：2+2=4）はパターンとして高確率で当てられる。
1. あまり見ない桁の掛け算（123×456）は厳密計算ではなく「それっぽい続きの数字」を予測するため外れやすい。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/01-first-llm.html#ex-ch01-q11

</details>

*(この節の追加問題 `ch01-q12` はWeb教科書 https://sakunokyumei.github.io/llm-research-textbook/01-first-llm.html で自動採点・復習できます)*

## チャット＝台本の続き

**ミナト**：でも、ただ「続きを書いているだけ」なら、どうして質問に答えるチャットができるんですか？

**ボンゴ先生**：いい質問だ！ 実はチャットも「続きを書いている」だけなんだ。さっきの `messages`（辞書のリスト）を、`chat.tokenizer` が裏で**「台本」の形の1本の文章**に変換してからモデルに渡している。中身をのぞいてみよう。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
messages = [
    {"role": "system", "content": "あなたは陽気な物理の先生です。"},
    {"role": "user", "content": "空はなぜ青いの？"},
]
script = chat.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(script)

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
<|im_start|>system
あなたは陽気な物理の先生です。<|im_end|>
<|im_start|>user
空はなぜ青いの？<|im_end|>
<|im_start|>assistant
```

</details>

この1行のドットつなぎを分解して読んでみましょう。

- `chat`：`pipeline` が作ったオブジェクト
- `.tokenizer`：`chat` の中に入っている「文章⇔数字・台本変換の担当オブジェクト」（属性）
- `.apply_chat_template(...)`：`tokenizer` が持つメソッド
  - `messages`：会話のリスト（入力）
  - `tokenize=False`：まだ数字のIDに変えず、人間が読める文字列（`str`）のまま返してほしいという指定
  - `add_generation_prompt=True`：末尾に `<|im_start|>assistant`（ここからアシスタントのセリフ開始）を付け足す指定

> **💡 理論のポイント：台本の続きを書かせる仕組み**
>
> `<|im_start|>` と `<|im_end|>` は「セリフの始まり／終わり」を表す特別な記号です。末尾が `<|im_start|>assistant` で終わっているので、LLMは「次はアシスタントのセリフが来るはずだ」と予測して返事を書き始め、書き終えると `<|im_end|>` を予測して止まります。
>
> モデル名の `Instruct` は、この台本形式で適切に返答できるように追加学習（ファインチューニング、第18章で自作）された印です。

#### 📝 確認問題（`ch01-q13` · Lv.2 · チャットテンプレートと特殊トークンの役割）

`apply_chat_template(messages, tokenize=False, add_generation_prompt=True)` で末尾に `<|im_start|>assistant` を付けるのはなぜですか？

- **a.** この記号を末尾に付けないと、Pythonの標準出力が日本語を処理できず文字化けエラーを起こすから
- **b.** これまで入力されたユーザーの質問文を台本から削除して、モデルに見えないよう隠すため
- **c.** 特殊トークンを末尾に追加することで、モデル本体のGPUメモリ（VRAM）消費量を半分に圧縮できるから
- **d.** LLMは文章の続きを書く仕組みなので、「ここからAIの返答」と台本で合図するため

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：チャット用LLMも根本は「入力された文字列の続きを予測するモデル」であることを思い出しましょう。
- **ヒント 2**：もし末尾に `<|im_start|>assistant` がなければ、モデルは誰のセリフの続きを書いてしまうでしょうか？
- **正解**：`d`

**考え方**：チャットモデルも本質は次トークン予測器です。会話リストを台本文字列に変換し、末尾をアシスタントの発言開始印にすることで「続き＝AIの返答」を生成させます。

1. `messages`（辞書のリスト）は人間とPythonにとって扱いやすい形式。
1. `apply_chat_template` がそれを1本の台本文字列に変換する。
1. `add_generation_prompt=True` で末尾に `<|im_start|>assistant\n` を付け、その続きを書かせる。
1. 返答が終わるとモデル自身が `<|im_end|>` を予測して停止する。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/01-first-llm.html#ex-ch01-q13

</details>

## キャラクターを演じさせる

台本の一番上の `system`（役の設定）を変えると、AIの口調やルールを自在に変えられます。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
def ask_as(character, question, max_new_tokens=200):
    messages = [
        {"role": "system", "content": character},
        {"role": "user", "content": question},
    ]
    result = chat(messages, max_new_tokens=max_new_tokens)
    return result[0]["generated_text"][-1]["content"]

bongo = "あなたはボンゴ先生です。陽気な物理の先生で、答えに必ずダジャレを1つ入れます。"
print(ask_as(bongo, "空はなぜ青いの？"))

<details><summary><b>📊 出力の例（数値や文章は実行ごとに変わってOK）</b>（実行後に自分の出力と見比べよう）</summary>

```text
いい質問だね！空が青いのは、太陽の光が空気の粒にぶつかって散らばるからなんだ。青い光は波が短くて散らばりやすいから、空いっぱいに青が広がるんだよ。これを「レイリー散乱」と言うんだ。散乱だけに、さんざん考えた答えだよ！
```

</details>

> **🔬 やってみよう：systemを書き換えよう**
>
> `bongo` の中身を変えて、同じ質問に対する答えの変化を観察してみましょう。
>
> - 「あなたは江戸時代の侍です。古風な言葉で答えます。」
> - 「あなたは5歳の子どもにも分かるように説明する先生です。」
> - 「あなたは何を聞かれても『それはLLMの得意分野です』と答えます。」
>
> 0.5Bの小さなモデルなので指示を忘れることもあります。第18〜19章では、`system` に書かなくてもキャラクターを体得させる**学習（SFT・LoRA）**を行います。

#### 📝 確認問題（`ch01-q14` · Lv.3 · systemロールとuserロールの使い分け）

LLMに「すべての質問に英語だけで、3文以内で答える家庭教師」を演じさせつつ、毎回の質問内容だけを変えたいとき、もっとも適切な `messages` の構成はどれですか？

- **a.** `role: "assistant"` に利用者からの質問を書き、`role: "user"` にAI側のルールや返答を書く
- **b.** `role: "system"` に家庭教師のルールを書き、`role: "user"` に毎回の具体的な質問を書く
- **c.** `pipeline` の `model=` 引数に `"english-teacher"` と直接書いて役割を切り替える
- **d.** `max_new_tokens="english"` のように生成引数へ文字列を渡して出力言語を指定する

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：会話全体で変わらない「役割・制約」と、毎回変わる「質問内容」をどのロールに分けるか考えましょう。
- **ヒント 2**：本文の `ask_as(character, question)` が `system` と `user` に何を入れていたか思い出してください。
- **正解**：`b`

**考え方**：`system` は会話全体の方針・役割設定、`user` はその回の具体的な質問という分離をしておくと関数化しやすく再利用できます。

1. 変わらない前提ルール（性格・出力形式）→ `{"role": "system", "content": ...}`。
1. 毎回変わる質問 → `{"role": "user", "content": ...}`。
1. `ask_as(character, question)` のように引数で分けると使い回せる。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/01-first-llm.html#ex-ch01-q14

</details>

## 「大規模」ってどれくらい？

LLMの「L」は Large（大規模）。何が大規模なのでしょう？ モデルの中にある「調整できる数（**パラメータ**）」を数えてみます。

まず、本物の1行コードを見る前に、**理解のための最小コード**で「合計を数える2通りの書き方」を見比べましょう。

<details><summary><b>基礎の補足：理解のための最小コード：forループと sum(... for ...) の対応</b>（クリックで開く）</summary>

モデルのパラメータは、たくさんの部品（層ごとの数値の表）に分かれて保存されています。全部で何個あるか数えるには、部品を1つずつ取り出して要素数を足せばOKです。

```python no-run
# 仮に3つの部品（要素数が 4個、6個、2個）からなる小さなモデルだとする
fake_params = [[0.1, 0.2, 0.3, 0.4], [0.5, 0.6, 0.7, 0.8, 0.9, 1.0], [1.1, 1.2]]

# 書き方A：ふつうの for 文（中間値が見えて分かりやすい）
total = 0
for p in fake_params:
    count = len(p)          # この部品の要素数（4 → 6 → 2）
    total += count          # 合計に足す（4 → 10 → 12）
print("for文の合計：", total)        # 12

# 書き方B：sum とジェネレータ式で1行にしたもの（中身はAとまったく同じ！）
total2 = sum(len(p) for p in fake_params)
print("1行版の合計：", total2)       # 12
```

本物のPyTorchモデルでは、`fake_params` にあたるのが `chat.model.parameters()`、要素数を返す `len(p)` にあたるのが **`p.numel()`**（number of elements＝全要素数）です。

</details>

では本物のモデルで数えてみましょう！

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
n = sum(p.numel() for p in chat.model.parameters())
print(f"パラメータの数：{n:,} 個")

<details><summary><b>📊 出力の例（数値や文章は実行ごとに変わってOK）</b>（実行後に自分の出力と見比べよう）</summary>

```text
パラメータの数：494,032,768 個
```

</details>

`f"…{n:,}…"` は **f文字列**で、`{n}` に変数の値を埋め込み、`:,` を付けると3桁ごとにカンマが入ります（`494032768` → `494,032,768`）。

> **💡 理論のポイント：パラメータ＝調整できるつまみ、そして「B」の意味**
>
> **パラメータ**は、モデルの中にある「学習で調整する数値」です。音楽ミキサーに並ぶつまみのようなもので、LLMの学習とは「次の言葉当てゲームの成績が良くなるように、このつまみを少しずつ回す作業」です。
>
> ここで指数の書き方を1つ確認しておきましょう。
> - $10^9$（「じゅうのきゅうじょう」と読む）＝ 10を9回掛けた数 ＝ `1,000,000,000`（**10億** ＝ 1 **B**illion）
> - このモデルは約4.94億個 $\approx 0.5 \times 10^9$ 個なので、名前が **`0.5B`** です。
> - モデルのメモリ容量の見積もり：**パラメータ数 × 1個あたりのバイト数**。`float16`（16ビット＝2バイト）なら約4.94億 × 2バイト $\approx$ 約9.88億バイト（約988MB）。さっきダウンロードしたモデル重みファイル（`model.safetensors: 988M`）とぴったり一致します！
>
> なお、`nvidia-smi` に出た `MiB` は2進法基準（$2^{20} = 2$ を20回掛けた数 $= 1,048,576$ バイト）で、10進法の `MB`（$10^6 = 1,000,000$ バイト）とは少しだけ違います。

確認問題と自力実装、さらにVRAM見積もりの発展問題に挑戦しましょう！

#### 📝 確認問題（`ch01-q15` · Lv.3 · forループとsum(ジェネレータ式)の対応）

パラメータ数を数える1行コード `sum(p.numel() for p in ...)` の仕組みを確認します。次のコードの出力を予測してください。

```python
parts = [[1, 2, 3], [4, 5], [6, 7, 8, 9]]
n = sum(len(p) for p in parts)
print(n)
```

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：`sum(len(p) for p in parts)` は、`parts` の各要素の長さを順に取り出して合計する書き方です。
- **ヒント 2**：3つのリスト `[1, 2, 3]`, `[4, 5]`, `[6, 7, 8, 9]` の要素数をそれぞれ数えて足し合わせましょう。
- **期待される出力**：`9`

**考え方**：`sum(処理 for p in 集まり)` は、`for` 文で1つずつ計算した値を合計する短縮表現です。

1. 1つ目の `p = [1, 2, 3]` → `len(p) = 3`
1. 2つ目の `p = [4, 5]` → `len(p) = 2`
1. 3つ目の `p = [6, 7, 8, 9]` → `len(p) = 4`
1. 合計：$3 + 2 + 4 = 9$

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/01-first-llm.html#ex-ch01-q15

</details>

#### 📝 確認問題（`ch01-q16` · Lv.4 · 【自力実装】for文でパラメータ総数を数える関数）

各層（リスト）のリストを受け取り、中に入っている数値の総数を `for` 文（または `sum`）で数えて返す関数 `count_params(layers)` を自力実装し、Colabで検算テストを通してください。

**ひな形コード：**
```python
def count_params(layers):
    # layers は [[0.1, 0.2], [0.3, 0.4, 0.5]] のようなリストのリスト
    # 全体の要素数を整数で返す
    pass
```

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：合計を入れる変数 `total = 0` を用意し、`layers` を `for` 文で1つずつ取り出します。
- **ヒント 2**：各 `layer` の要素数は `len(layer)` で取得できるので、`total += len(layer)` で加算して最後に返します。

**解答例：**
```python
def count_params(layers):
    total = 0
    for layer in layers:
        total += len(layer)
    return total
```

**検算テスト（Colabで実行して確認）：**
```python
assert count_params([[1, 2], [3, 4, 5]]) == 5
assert count_params([]) == 0
assert count_params([[0.1] * 10, [0.2] * 25, [0.3] * 5]) == 40
print("ALL PASSED")
```

**考え方**：モデルの全パラメータ数を数える処理は、「各層の要素数を順に取り出して足す」という単純なループです。

1. `total = 0` で合計用の変数を初期化する。
1. `for layer in layers:` で各層を取り出す。
1. `total += len(layer)` で要素数を加算し、最後に `return total`。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/01-first-llm.html#ex-ch01-q16

</details>

*(この節の追加問題 `ch01-q17`, `ch01-q18` はWeb教科書 https://sakunokyumei.github.io/llm-research-textbook/01-first-llm.html で自動採点・復習できます)*

## うまく動かないときは

> **⚠️ つまずき注意：よくあるつまずき**
>
> - **`ModuleNotFoundError: No module named 'transformers'`**：Colab以外（手元のPC）で実行していませんか？ Colabには最初から入っています。ローカル環境は[付録A](https://sakunokyumei.github.io/llm-research-textbook/a-setup.html)へ。
> - **`CUDA out of memory`**：GPUのメモリ（VRAM）が足りません。「ランタイム」→「セッションを再起動する」を押して、上から実行し直してください。
> - **途中で接続が切れた**：無料版Colabは放置すると数十分〜1時間半ほどで切断されます。「再接続」して上から実行し直せばOKです（コードは残ります）。
> - **ダウンロードが遅い・失敗する**：少し時間をおいてそのセルだけ再実行してください。

**✅ できたらチェック**

- [ ] Colabでノートブックを作り、T4 GPU（またはCPU）でLLMを動かした
- [ ] モジュール・オブジェクト・属性・メソッド・キーワード引数の違いを説明できる
- [ ] `result[0]["generated_text"][-1]["content"]` の各段階の型と意味を説明でき、取り出し関数を自力実装した
- [ ] `for` ループと `sum(p.numel() for p in ...)` の対応を理解し、パラメータ数とメモリ容量を見積もれた
- [ ] 「LLMは次の言葉（トークン、第2章）の確率予測機であり、チャットも台本の続きを書かせている」と説明できる

> **📝 まとめ**
>
> - `pipeline(...)` はトークナイザー・モデル・生成手順をまとめたオブジェクトを作る。`chat.model` や `chat.tokenizer` はその中の部品（属性）
> - 会話の返り値はリストと辞書の入れ子。最新のAI返答は常に末尾 `[-1]` にある
> - LLMがやっているのは「次の言葉の確率予測」の繰り返し。チャットも `apply_chat_template` で台本にして続きを書かせている
> - `0.5B` は約 $0.5 \times 10^9$（約5億）個のパラメータの意味。必要メモリは「パラメータ数 × 1個あたりのバイト数」で見積もれる

**ボンゴ先生**：第1章おつかれさま！ ただ動かしただけでなく、コードの箱の中身までしっかりのぞけたね。次の第2章では、LLMが日本語をどうやって「数字の列（トークン）」に変えているのかを分解して、最後には自分の手でトークナイザーを作ってみよう！

## 学習のふりかえり・次のステップ

- **困ったとき・エラーが出たとき**：[エラー辞典（付録E）](https://sakunokyumei.github.io/llm-research-textbook/e-errors.html)
- **忘れる前に定着を確認**：[学習状況・間隔復習ダッシュボード](https://sakunokyumei.github.io/llm-research-textbook/review.html) で確認問題・類題・7段階の進捗を記録できます
- **この章のWebページ**：https://sakunokyumei.github.io/llm-research-textbook/01-first-llm.html
- **次の章**：[第2章 文章を数字にする：トークン](https://sakunokyumei.github.io/llm-research-textbook/02-tokens.html)